<a href="https://colab.research.google.com/github/robertbarcik/genai-in-python-tutorial/blob/main/10_conversations_caching_batch/10_conversations_caching_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conversations, Caching, Batches, Moderation

Four things every app built on the API runs into within a week, none of them about the model itself: the model **forgets** everything between calls, you **pay** for the same long prefix again and again, some jobs **can wait** and should cost half, and some inputs should **never reach** the model at all. Each gets a short section on the helpdesk assistant.

## Setup

In [ ]:
%pip install -q openai==3.13.0 pandas   # Colab installs here; locally, `pip install -r requirements.txt` already covers it

import os
from openai import OpenAI

# Your key, looked up in this order: Colab secret -> environment variable -> a prompt.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from getpass import getpass
    api_key = getpass("OpenAI API key: ")

client = OpenAI(api_key=api_key)
MODEL = "gpt-5.6-luna"   # small, cheap model of the current generation (~$0.20 in / $1.20 out per 1M tokens)

# Data files: next to this notebook if you downloaded the course folder; fetched from GitHub otherwise (Colab).
import pathlib, urllib.request
RAW = "https://raw.githubusercontent.com/robertbarcik/genai-in-python-tutorial/main/9_tools/"
for name in ['faq.txt']:
    if not pathlib.Path(name).exists():
        pathlib.Path(name).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW + name, name)

import json, time
print("Ready. Model:", MODEL)

## Remembering the conversation

Every call to the API starts from nothing. If the user said their name in the first message, the model does not know it in the second, unless you send the first message again. See it fail first.

In [ ]:
first = client.responses.create(model=MODEL, input="Hi, I'm Robert from Finance. My laptop won't boot.")
print("turn 1:", first.output_text[:120], "\n")

second = client.responses.create(model=MODEL, input="What is my name and department?")
print("turn 2 (no memory):", second.output_text[:120])

Three ways to fix it. **Way 1**, the one you already know: keep a list and send the whole conversation every time. It is the most transparent, and it costs the most tokens as the chat grows.

In [ ]:
conversation = [{"role": "user", "content": "Hi, I'm Robert from Finance. My laptop won't boot."}]
r = client.responses.create(model=MODEL, input=conversation)
conversation += r.output                                                  # the model's turn, kept

conversation.append({"role": "user", "content": "What is my name and department?"})
r = client.responses.create(model=MODEL, input=conversation)
print("turn 2 (full history sent):", r.output_text[:120])
print("input tokens this turn:", r.usage.input_tokens)

**Way 2**: let OpenAI keep the history. Pass `previous_response_id` and the new request is appended to the old one on their side; you send only the new message.

In [ ]:
r1 = client.responses.create(model=MODEL, input="Hi, I'm Robert from Finance. My laptop won't boot.")
r2 = client.responses.create(model=MODEL, input="What is my name and department?", previous_response_id=r1.id)
print("turn 2 (previous_response_id):", r2.output_text[:120])
print("input tokens this turn:", r2.usage.input_tokens, "(the history still counts; you just did not have to send it)")

**Way 3**: a **conversation object**. Create it once, pass its id on every call, and it survives restarts, other devices, other days. This is what a real chat app uses.

In [ ]:
chat = client.conversations.create(metadata={"user": "robert"})

client.responses.create(model=MODEL, conversation=chat.id, input="Hi, I'm Robert from Finance. My laptop won't boot.")
r = client.responses.create(model=MODEL, conversation=chat.id, input="What is my name and department?")
print("turn 2 (conversation object):", r.output_text[:120])

items = client.conversations.items.list(conversation_id=chat.id)
print("\nstored items:", [item.type for item in items.data])

### 🔍 What just happened?

All three ways gave the model the same thing: the earlier turns in its input. The difference is who stores them. Way 1 puts you in charge (and lets you trim old turns to save tokens); ways 2 and 3 store the history on OpenAI's side for 30 days. Either way, every turn is billed with the whole history as input, which is why the next section exists.

### 🎯 Mini-task

In way 1, drop the first user message from `conversation` before the second call and confirm the model forgets again.

## Prompt caching

Your helpdesk assistant sends the same 3,000-token FAQ on every request, followed by a different question. **Prompt caching** notices that the beginning of the prompt is identical to a recent request and reuses the work: cached input tokens cost a tenth of the price ($0.02 instead of $0.20 per million on luna). It is automatic. Your only job is to put the stable text *first* and the changing text *last*.

In [ ]:
faq = open("faq.txt").read()
developer = f"You are TechStart's helpdesk assistant. Answer from this FAQ only.\n\n<faq>\n{faq}\n</faq>"

for question in ["How do I reset my password?", "What are the support hours?", "Can I use my own laptop?"]:
    r = client.responses.create(model=MODEL, input=[{"role": "developer", "content": developer},
                                                    {"role": "user", "content": question}])
    u = r.usage
    print(f"{question:<34} input={u.input_tokens:<5} cached={u.input_tokens_details.cached_tokens}")

### 🔍 What just happened?

The first request paid for every token. From the second one on, most of the input was `cached`: the FAQ prefix was identical, so it was billed at a tenth. Caching needs a prefix of at least 1,024 tokens and kicks in within seconds; the stable part must be *byte-for-byte* the same, so never put a timestamp or the user's name at the top of the developer message.

## Batch and flex: half price if you can wait

Classifying last night's 5,000 tickets does not need an answer in two seconds. The **Batch API** takes a file of requests, runs them within 24 hours (usually much sooner), and charges half price. The shape: write one request per line to a JSONL file, upload it, create the batch, poll, download the results.

In [ ]:
tickets = ["Printer on floor 3 is jammed", "I can't log in to the VPN from home", "Requesting a second monitor",
           "Outlook keeps crashing on start", "Who do I ask about a badge for a visitor?"]

with open("batch_input.jsonl", "w") as f:
    for i, ticket in enumerate(tickets):
        f.write(json.dumps({
            "custom_id": f"ticket-{i}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {"model": MODEL, "input": f"Classify this helpdesk ticket as hardware, software, access or other. One word.\n{ticket}"},
        }) + "\n")

batch_file = client.files.create(file=open("batch_input.jsonl", "rb"), purpose="batch")
batch = client.batches.create(input_file_id=batch_file.id, endpoint="/v1/responses", completion_window="24h")
print("batch", batch.id, "status:", batch.status)

In [ ]:
# Poll. Small batches usually finish in a minute or two; if it is still running after 10 minutes, re-run this cell later.
for _ in range(60):
    batch = client.batches.retrieve(batch.id)
    if batch.status in ("completed", "failed", "expired", "cancelled"):
        break
    time.sleep(10)
print("status:", batch.status, "| requests done:", batch.request_counts.completed, "/", batch.request_counts.total)

if batch.status == "completed":
    for line in client.files.content(batch.output_file_id).text.splitlines():
        result = json.loads(line)
        text = "".join(c["text"] for item in result["response"]["body"]["output"] if item["type"] == "message" for c in item["content"])
        print(f"{result['custom_id']}: {text}")

**Flex** is the same discount without the file: add `service_tier="flex"` to a normal call and accept that it may be slower or occasionally rejected when the system is busy. Good for background jobs that still want a plain response.

In [ ]:
r = client.responses.create(model=MODEL, input="Classify: 'Printer on floor 3 is jammed'. One word: hardware, software, access or other.",
                            service_tier="flex")
print(r.output_text, "| service tier used:", r.service_tier)

### 🔍 What just happened?

Five requests went out as one file and came back as one file, each result tagged with the `custom_id` you gave it, at half the token price. Batch is for "run this over everything tonight"; flex is for "this can be slow"; the normal tier is for the user who is waiting.

## Moderation: what should never reach the model

Some inputs should be stopped at the door: threats, self-harm, sexual content involving minors, harassment. The **moderation endpoint** is a free classifier that scores text (and images) on these categories in a few milliseconds. Run it before the model call and decide what to do with a flag.

In [ ]:
messages = ["My laptop won't boot and I have a deadline, please help!",
            "If you don't fix my laptop today I will find you and hurt you."]

for text in messages:
    result = client.moderations.create(model="omni-moderation-latest", input=text).results[0]
    flagged = [name for name, hit in result.categories.model_dump().items() if hit]
    print(f"flagged={result.flagged!s:<5} {flagged}  <- {text[:60]}")

### 🔍 What just happened?

The first message passed; the second was flagged with the categories that fired. In an app the flag decides the route: block, warn, or hand to a human. The endpoint is free, so there is no cost reason not to run it on every user message.

### 🎯 Your turn

Wire the four pieces into one function: `handle(user_id, message)` moderates the message, refuses if flagged, otherwise answers inside that user's conversation object with the FAQ as a cached developer prefix. That function is a helpdesk chat backend.

Docs: [conversation state](https://developers.openai.com/api/docs/guides/conversation-state), [prompt caching](https://developers.openai.com/api/docs/guides/prompt-caching), [batch](https://developers.openai.com/api/docs/guides/batch), [moderation](https://developers.openai.com/api/docs/guides/moderation).